# Minimum Solar Farm Optimisation 

In [ ]:
# from time_series.ipynb 
forecasted_demand_2027 = 23241.7
forecasted_demand_2028 = 23924.8 
forecasted_demand_2029 = 24645.1
forecasted_demand_2030 = 25346.3

current_re_penetration = 0.087 # 8.7%

# solar farm specifications 
farm_capacity_mw = 75 # MW installed per utility scale farm
capacity_factor = 0.25 # 25% average for SA
avg_farm_output_mw = farm_capacity_mw * capacity_factor 
farms = 8

print("Solar Farm Specifications:")
print(f"Installed capacity per farm: {farm_capacity_mw} MW")
print(f"Capacity factor: {capacity_factor * 100:.0f}%")
print(f"Average output per farm: {avg_farm_output_mw} MW")

## Energy Gap

In [ ]:
import math 

for year, demand in [(2027, forecasted_demand_2027), (2028, forecasted_demand_2028), (2029, forecasted_demand_2029), (2030, forecasted_demand_2030)]:
    existing_re_output = demand * current_re_penetration
    energy_gap = demand - existing_re_output
    
    # min farms needed
    farms_needed = math.ceil(energy_gap / avg_farm_output_mw)
    utility_scale_pct = 0.6 # flat provinces
    small_scale_pct = 0.4 # constrained provinces (KZN, EC) 
    
    utility_farms = math.ceil(farms_needed * utility_scale_pct)
    small_scale_farms = math.ceil(farms_needed * small_scale_pct)
    
    print("=" * 55)
    print(f"{year} ANALYSIS")
    print("=" * 55)
    print(f"Forecasted Demand: {demand:.1f} MW")
    print(f"Existing Renewable Energy Output (8.7%): {existing_re_output:.1f} MW")
    print(f"Energy Gap to Fill: {energy_gap:.1f} MW")
    print(f"Average Output Per Farm: {avg_farm_output_mw} MW")
    print(f"Total Farms Needed: {farms_needed}")
    print("\nRecommended Split:")
    print(f"Utility Scale Farms: {utility_farms}")
    print(f"Small Scale Farms: {small_scale_farms}")
    print(f"\nTotal Capacity if Built: {farms_needed * avg_farm_output_mw:.1f} MW")
    print(f"Renewable Energy Penetration After Build: {((existing_re_output + farms_needed * avg_farm_output_mw) / demand * 100):.1f}%")

### Realistic Renewable Energy Target 

In [ ]:
target_re_penetration = 0.25 # 25% by 2030

for year, demand in [(2027, forecasted_demand_2027), (2028, forecasted_demand_2028), (2029, forecasted_demand_2029), (2030, forecasted_demand_2030)]:
    target_re_output = demand * target_re_penetration
    gap_to_fill = target_re_output - existing_re_output 
    farms_needed = math.ceil(gap_to_fill / avg_farm_output_mw)
    utility_farms = math.ceil(farms_needed * 0.6)
    small_scale_farms = math.ceil(farms_needed * 0.4)
    
    print("=" * 55)
    print(f"{year} ANALYSIS")
    print("=" * 55)
    print(f"Forecasted Demand: {demand:.1f} MW")
    print(f"Current Renewable Energy Output (8.7%): {existing_re_output:.1f} MW")
    print(f"Target Renewable Energy Output (25%): {target_re_output:.1f}")
    print(f"Energy Gap to Fill: {gap_to_fill:.1f} MW")
    print(f"Total Farms Needed: {farms_needed}")
    print("\nRecommended Split:")
    print(f"Utility Scale Farms (60%): {utility_farms}")
    print(f"Small Scale Farms (40%): {small_scale_farms}")
    print(f"\nTotal Renewable Energy Capacity: {farms_needed * avg_farm_output_mw:.1f} MW")
    print(f"Renewable Energy Penetration After Build: {((existing_re_output + farms_needed * avg_farm_output_mw) / demand * 100):.1f}%")
    print("\nPhased Implementation:")
    print("Phase 1 (Priority Locations Identified): 8 farms")
    print(f"Phase 2 (Remaining Utility Scale): {utility_farms - farms} farms")
    print(f"Phase 3 (Small Scale & Rooftop): {small_scale_farms} farms\n")

## Save the Ouputs

In [ ]:
import math 
import pandas as pd

# build summary
rows = []
annual_all = pd.read_csv("annual_demand_forecast.csv")

for _, row in annual_all[annual_all["year"] >= 2027].iterrows():
    year = int(row["year"])
    demand = round(row["avg_demand_mw"], 1)
    current_re = round(demand * current_re_penetration, 1)
    target_re = round(demand * target_re_penetration, 1)
    gap = round(target_re - current_re, 1)
    farms_needed = math.ceil(gap / avg_farm_output_mw)
    utility_farms = math.ceil(farms_needed * 0.6)
    small_farms = math.ceil(farms_needed * 0.4)
    total_capacity = round(farms_needed * avg_farm_output_mw, 1)
    re_after = round(((current_re + total_capacity) / demand) * 100, 1)

    rows.append({
        "year": year,
        "forecasted_demand_mw": demand,
        "current_re_output_mw": current_re,
        "target_re_output_mw": target_re,
        "energy_gap_mw": gap,
        "total_farms_needed": farms_needed,
        "utility_scale_farms": utility_farms,
        "small_scale_farms": small_farms,
        "priority_farms": farms,
        "utility_farms": utility_farms - farms,
        "small_farms": small_farms - farms,
        "total_re_capacity_mw": total_capacity,
        "re_penetration_after_pct": re_after
    })

# add to DataFrame
optimisation_summary = pd.DataFrame(rows)

optimisation_summary.to_csv("optimisation_summary.csv", index = False)
print("Saved: optimisation_summary.csv")
print(optimisation_summary.to_string(index = False))